# TDNEGF current workflow guide (well-scoped, function-by-function)

This notebook is a **developer guide for the current implementation**.
It intentionally documents the block backend that is actively used in repository examples.

## Source-of-truth policy used here
When there is disagreement between old material and current code, this notebook follows:
1. `README.md`
2. active examples in `examples/`
3. current package implementation in `src/`

`test/TDNGF_V3.ipynb` is used only as a **legacy reference** for mapping and migration context.


## Working plan (done before writing)

1. Inspect current README/examples/src to identify the recommended workflow.
2. Compare that workflow against `test/TDNGF_V3.ipynb` function-by-function.
3. Write one implementation-faithful notebook with:
   - architecture,
   - data structures,
   - propagation path,
   - observables/post-processing,
   - practical old-to-new mapping,
   - maintenance pitfalls and audit notes.
4. Keep changes minimal (documentation-only) and avoid API refactors.


## Evidence summary: what is current right now

- `README.md` explicitly marks block backend as recommended (`eom_tdnegf_blocks!`).
- `examples/01_two_terminal_square_lattice.jl` and `examples/04_blocks_1d_twochannel_benchmark.jl` use:
  - `SelfEnergyBlock`,
  - `ExperimentalBlockRHSParams`,
  - `eom_tdnegf_blocks!`,
  - `pointer_blocks(...)` + block-native observables.
- `src/eom_tdnegf.jl` contains the block RHS implementation and block param struct.
- `src/observables.jl` includes block-native `obs_Ixα!(ptr, p_blocks, obs)`.


In [ ]:
using Pkg

# Robust activation: prefer repository root relative to notebook location.
# Fallback to current working directory when __DIR__ is unavailable.
project_root = try
    normpath(joinpath(@__DIR__, "..", ".."))
catch
    pwd()
end
Pkg.activate(project_root)

using TDNEGF
using DifferentialEquations
using LinearAlgebra
using Statistics

println("Activated project: ", Base.active_project())

## 1) Architecture overview (current)

Current package ships two RHS backends:

- **Recommended (current)**: heterogeneous block backend
  - RHS: `eom_tdnegf_blocks!`
  - params: `ExperimentalBlockRHSParams`
  - block definition: `SelfEnergyBlock`
- **Legacy compatibility**: rectangular backend
  - RHS: `eom_tdnegf!`
  - params: `ModelParamsTDNEGF`

### Practical high-level flow
1. Build lattice/model metadata (`ModelParamsTDNEGF`) and `H_ab`.
2. Build pole/self-energy/coupling ingredients (`Σ`, `χ`, `ξ`).
3. Build `SelfEnergyBlock` objects and `Δ_blocks`.
4. Create `ExperimentalBlockRHSParams(H_ab, blocks, Δ_blocks, p_model)`.
5. Create `u0` with block layout size.
6. Solve with `ODEProblem(eom_tdnegf_blocks!, ...)`.
7. Post-process with `pointer_blocks(...)` and observable APIs.


## 2) Function-by-function mapping (V3 style → current API)

Below is the direct mapping in the spirit of `TDNGF_V3`.

| Legacy V3 function style | Current function/type to use | Notes |
|---|---|---|
| `pade_poles`, custom pole loading | `load_poles_square` (and internal pole helpers) | Use repository data tables through package API. |
| In-notebook `build_Σᴸ_nλ`, `build_Σᴳ_nλ`, `build_χ_nλ`, `build_ξ_an` | Same exported names from `TDNEGF` | Keep behavior centralized in `src/selfenergy.jl`. |
| In-notebook `build_H_ab` | exported `build_H_ab` | Same conceptual role; avoid duplicate local defs. |
| In-notebook `ModelParamsTDNEGF`, `DynamicalVariables`, `pointer` | package-defined structs/helpers in `src/types.jl` | No notebook-side struct redefinitions. |
| In-notebook `eom!` | `eom_tdnegf_blocks!` (recommended) / `eom_tdnegf!` (legacy) | Prefer block backend for new work. |
| In-notebook observables functions | `ObservablesTDNEGF`, `obs_n_i!`, `obs_σ_i!`, `obs_Ixα!` | Block path uses `pointer_blocks` and block-order current indexing. |

### Migration rule
If V3 defines a function with the same role, prefer the exported package function and keep notebook code as orchestration only.


## 3) Main data structures (what each one owns)

- `ModelParamsTDNEGF`
  - geometry sizes, local metadata, and general model containers.
- `SelfEnergyBlock`
  - **static** block definitions (`Nc`, `N_λ1`, `N_λ2`, `ΣL`, `ΣG`, `χ`, `ξ`).
- `ExperimentalBlockRHSParams`
  - **dynamic/runtime** solver context (`H_ab`, `Δ_blocks`, caches, aux layout).
- `pointer_blocks(u, dims_ρ_ab, aux_layout)`
  - typed views from flattened state into `ρ`, per-block `Ψ`, and per-pair `Ω` sectors.

This static-vs-dynamic split is central for maintainability and parameter scans.


In [ ]:
function init_params_blocks_demo(; Nx::Int=6, Ny::Int=2, Nσ::Int=2, N_orb::Int=1,
                                  N_λ1::Int=49, N_λ2::Int=20,
                                  β::Float64=33.0, γ::Float64=1.0,
                                  γso::ComplexF64=0.5 + 0im,
                                  Vbias::Float64=1.0)
    # 1) Pole data
    Rλ, zλ = load_poles_square(N_λ1, N_λ2)

    # 2) Base model metadata
    p_model = ModelParamsTDNEGF(Nx=Nx, Ny=Ny, Nσ=Nσ, N_orb=N_orb,
                                Nα=2, N_λ1=N_λ1, N_λ2=N_λ2)

    # 3) Device Hamiltonian
    H_ab = build_H_ab(; Nx=p_model.Nx, Ny=p_model.Ny, Nσ=p_model.Nσ,
                      N_orb=p_model.N_orb, γ=γ, γso=γso)
    p_model.H_ab .= H_ab
    p_model.H0_ab .= H_ab

    # 4) Lead ingredients
    ΣL_nλ = build_Σᴸ_nλ(Rλ, zλ, p_model.Ny, p_model.Nσ, p_model.N_orb,
                       p_model.N_λ1, p_model.N_λ2; β=β, γ=γ)
    ΣG_nλ = build_Σᴳ_nλ(Rλ, zλ, p_model.Ny, p_model.Nσ, p_model.N_orb,
                       p_model.N_λ1, p_model.N_λ2; β=β, γ=γ)
    χ_nλ = build_χ_nλ(zλ, p_model.Ny, p_model.Nσ, p_model.N_orb,
                     p_model.N_λ1, p_model.N_λ2; β=β, γ=γ)

    ξ_anL = build_ξ_an(p_model.Nx, p_model.Ny, p_model.Nσ, p_model.N_orb;
                       xcol=1, y_coup=1:p_model.Ny)
    ξ_anR = build_ξ_an(p_model.Nx, p_model.Ny, p_model.Nσ, p_model.N_orb;
                       xcol=p_model.Nx, y_coup=1:p_model.Ny)

    # 5) Static blocks + dynamic shifts
    left_block  = SelfEnergyBlock(:left,  p_model.Nc, p_model.N_λ1, p_model.N_λ2,
                                  ΣL_nλ, ΣG_nλ, χ_nλ, ξ_anL)
    right_block = SelfEnergyBlock(:right, p_model.Nc, p_model.N_λ1, p_model.N_λ2,
                                  ΣL_nλ, ΣG_nλ, χ_nλ, ξ_anR)
    blocks = [left_block, right_block]
    Δ_blocks = ComplexF64[+Vbias/2 + 0im, -Vbias/2 + 0im]

    # 6) Preferred constructor carries observable geometry metadata
    p_blocks = ExperimentalBlockRHSParams(p_model.H_ab, blocks, Δ_blocks, p_model)

    # 7) Flattened state vector size from current block layout
    u0 = zeros(ComplexF64, p_blocks.dims_ρ_ab[1]^2 + p_blocks.aux_layout.total_size)
    return p_model, p_blocks, u0
end

p_model, p_blocks, u0 = init_params_blocks_demo();

println("Ns = ", p_model.Ns)
println("N_sites = ", p_model.N_sites)
println("N_blocks = ", length(p_blocks.blocks))
println("ρ dims = ", p_blocks.dims_ρ_ab)
println("aux size = ", p_blocks.aux_layout.total_size)

ptr0 = pointer_blocks(u0, p_blocks.dims_ρ_ab, p_blocks.aux_layout)
println("ptr ρ size = ", size(ptr0.ρ_ab))
println("ptr first block Ψ size = ", size(ptr0.blocks[1].Ψ_anλ))

## 4) Propagation workflow (block backend)

This is the current executable pattern used by active examples:

1. Construct `prob = ODEProblem(eom_tdnegf_blocks!, u0, tspan, p_blocks)`.
2. `solve(...)` with chosen tolerances.
3. Iterate over saved states for observables.

> Runtime note: larger `Nx`, `Ny`, `N_λ1`, `N_λ2` quickly increase cost. Use moderate values first.


In [ ]:
tspan = (0.0, 5.0)
prob = ODEProblem(eom_tdnegf_blocks!, u0, tspan, p_blocks)

sol = solve(prob, Vern7();
            saveat=0.5,
            adaptive=true,
            dense=false,
            reltol=1e-6,
            abstol=1e-8)

println("saved steps = ", length(sol.t), " | final time = ", sol.t[end])

## 5) Observables and post-processing

### What this stage does
- Creates `ObservablesTDNEGF` buffers.
- Rebuilds typed state views via `pointer_blocks(...)` per time slice.
- Computes local charge/spin and block-indexed charge/spin currents.

### Important indexing caveat
`obs.Iα` and `obs.Iαx` are indexed by **block order** (`p_blocks.blocks`), not automatically by physical-lead label unless your block construction is one-block-per-lead.


In [ ]:
obs = ObservablesTDNEGF(p_model; N_tmax=length(sol.t), N_leads=length(p_blocks.blocks))
obs.t .= sol.t

for (it, ut) in enumerate(sol.u)
    obs.idx = it
    ptr = pointer_blocks(ut, p_blocks.dims_ρ_ab, p_blocks.aux_layout)

    obs_n_i!(ptr, p_model, obs)
    obs_σ_i!(ptr, p_model, obs)
    obs_Ixα!(ptr, p_blocks, obs)
end

println("obs.n_i shape  = ", size(obs.n_i))
println("obs.σx_i shape = ", size(obs.σx_i))
println("obs.Iα shape   = ", size(obs.Iα))
println("obs.Iαx shape  = ", size(obs.Iαx))

# Light sanity values useful during debugging
qtot = vec(sum(obs.n_i; dims=1))
println("total charge (first,last) = ", (qtot[1], qtot[end]))

## 6) Legacy vs current workflow (compact practical mapping)

### Concepts preserved
- One-time ODE propagation of reduced density + auxiliaries.
- Pole-expanded lead embedding ingredients.
- Local density/spin and lead/block current observables.

### What changed in implementation practice
- Main RHS moved from V3-style in-notebook `eom!` to package `eom_tdnegf_blocks!`.
- Auxiliary layout moved from rectangular assumptions to heterogeneous block/pair layout.
- Current indexing semantics are now explicitly tied to block ordering.

### What is no longer recommended
- Redefining core structs/functions inside notebooks as the primary workflow.
- Treating legacy notebook internals as authoritative over `src/` and active `examples/`.


## 7) Light audit: what was checked and what was fixed

### Outdated / mismatched patterns seen in `test/TDNGF_V3.ipynb`
- Monolithic notebook includes many local redefinitions of package-level logic.
- Mixed experimental scopes (transport + unrelated testbeds) obscure canonical setup.
- Legacy naming/index assumptions can be misread when using block backend.

### Missing explanations that this notebook now adds
- Static (`SelfEnergyBlock`) vs dynamic (`ExperimentalBlockRHSParams`) ownership split.
- Explicit block-layout pointer semantics and observable indexing behavior.
- Function-by-function old-to-new migration guidance.

### Minimal-fix policy
- Only documentation notebook was changed.
- No package API changes, no broad refactor, no renaming of core structures.


## 8) Maintenance pitfalls checklist

1. **Environment mismatch**: notebook launched from wrong folder can activate wrong project.
2. **Index meaning drift**: block order != physical lead naming unless explicitly mapped.
3. **Cost scaling surprises**: pole count and lattice size can inflate solve time quickly.
4. **Doc/API drift**: if examples or signatures change in `src/`, update this notebook immediately.
5. **Legacy confusion**: keep `TDNGF_V3` as reference, not authority for current APIs.
